<a href="https://colab.research.google.com/github/jaiswalakanksha22-glitch/Python-AI/blob/AI/Project7_Jaiswal_Akanksha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You are a data scientist in an entertainment company. You are given a MovieLens dataset. It includes around 1 million ratings from 6000 users on 4000 movies, along with some user features, movie genres. In addition, the timestamp of each user-movie rating is provided, which allows creating sequences of movie ratings for each user, as expected by the BST model.

You are asked to develop a Transformer-based recommendation system using the attached Jupyter Notebook and writing a script in Python, and running all the cells.

1. Download and prepare the dataset.

In [3]:
import os

os.environ["KERAS_BACKEND"] = "jax"  # or torch, or tensorflow

import math
from zipfile import ZipFile
from urllib.request import urlretrieve
import numpy as np
import pandas as pd

import keras
from keras import layers, ops
from keras.layers import StringLookup

First, let's download the movielens data.

The downloaded folder will contain three data files: users.dat, movies.dat, and ratings.dat.

In [4]:
urlretrieve("http://files.grouplens.org/datasets/movielens/ml-1m.zip", "movielens.zip")
ZipFile("movielens.zip", "r").extractall()

Then, we load the data into pandas DataFrames with their proper column names.

In [5]:
users = pd.read_csv(
    "ml-1m/users.dat",
    sep="::",
    names=["user_id", "sex", "age_group", "occupation", "zip_code"],
    encoding="ISO-8859-1",
    engine="python",
)

ratings = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    names=["user_id", "movie_id", "rating", "unix_timestamp"],
    encoding="ISO-8859-1",
    engine="python",
)

movies = pd.read_csv(
    "ml-1m/movies.dat",
    sep="::",
    names=["movie_id", "title", "genres"],
    encoding="ISO-8859-1",
    engine="python",
)

Here, we do some simple data processing to fix the data types of the columns.

In [6]:
users["user_id"] = users["user_id"].apply(lambda x: f"user_{x}")
users["age_group"] = users["age_group"].apply(lambda x: f"group_{x}")
users["occupation"] = users["occupation"].apply(lambda x: f"occupation_{x}")

movies["movie_id"] = movies["movie_id"].apply(lambda x: f"movie_{x}")

ratings["movie_id"] = ratings["movie_id"].apply(lambda x: f"movie_{x}")
ratings["user_id"] = ratings["user_id"].apply(lambda x: f"user_{x}")
ratings["rating"] = ratings["rating"].apply(lambda x: float(x))

Each movie has multiple genres. We split them into separate columns in the movies DataFrame.

In [7]:
genres = ["Action", "Adventure", "Animation", "Children's", "Comedy", "Crime"]
genres += ["Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical"]
genres += ["Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"]

for genre in genres:
    movies[genre] = movies["genres"].apply(
        lambda values: int(genre in values.split("|"))
    )


2. Transform the movie ratings data into sequences.

---



In [31]:
# 2. Transform the movie ratings data into sequences

# Sort ratings chronologically for each user
ratings = ratings.sort_values(by=["user_id", "unix_timestamp"])

# Sequence length
sequence_length = 4

# Store transformed data
user_ids_seq = []
movie_sequences = []
rating_sequences = []
target_movies = []

# Group interactions by user
grouped_ratings = ratings.groupby("user_id")

for user_id, group in grouped_ratings:

    movie_list = group["movie_id"].tolist()
    rating_list = group["rating"].tolist()

    # Skip users with too few interactions
    if len(movie_list) <= sequence_length:
        continue

    # Create sliding window sequences
    for i in range(len(movie_list) - sequence_length):

        input_movies = movie_list[i : i + sequence_length]

        input_ratings = rating_list[i : i + sequence_length]

        target_movie = movie_list[i + sequence_length]

        # Store results
        user_ids_seq.append(user_id)
        movie_sequences.append(input_movies)
        rating_sequences.append(input_ratings)
        target_movies.append(target_movie)

# Create final dataframe
sequence_data = pd.DataFrame({
    "user_id": user_ids_seq,
    "movie_sequence": movie_sequences,
    "rating_sequence": rating_sequences,
    "target_movie": target_movies,
})

# Display sample data
print(sequence_data.head())

# Total training samples
print("Total sequences created:", len(sequence_data))

  user_id                                    movie_sequence  \
0  user_1  [movie_3186, movie_1270, movie_1721, movie_1022]   
1  user_1  [movie_1270, movie_1721, movie_1022, movie_2340]   
2  user_1  [movie_1721, movie_1022, movie_2340, movie_1836]   
3  user_1  [movie_1022, movie_2340, movie_1836, movie_3408]   
4  user_1  [movie_2340, movie_1836, movie_3408, movie_2804]   

        rating_sequence target_movie  
0  [4.0, 5.0, 4.0, 5.0]   movie_2340  
1  [5.0, 4.0, 5.0, 3.0]   movie_1836  
2  [4.0, 5.0, 3.0, 5.0]   movie_3408  
3  [5.0, 3.0, 5.0, 4.0]   movie_2804  
4  [3.0, 5.0, 4.0, 5.0]   movie_1207  
Total sequences created: 976049


3. Define MetaData

In [32]:
# 3. Define metadata

# Unique users and movies
user_ids = sorted(ratings["user_id"].unique().tolist())
movie_ids = sorted(ratings["movie_id"].unique().tolist())

# Vocabulary sizes
num_users = len(user_ids)
num_movies = len(movie_ids)

# Model configuration
sequence_length = 4
embedding_dim = 32
num_heads = 4
dropout_rate = 0.1

# Store metadata
metadata = {
    "num_users": num_users,
    "num_movies": num_movies,
    "sequence_length": sequence_length,
    "embedding_dim": embedding_dim,
    "num_heads": num_heads,
    "dropout_rate": dropout_rate,
}

# Display metadata
print("Metadata:\n")

for key, value in metadata.items():
    print(f"{key}: {value}")

Metadata:

num_users: 6040
num_movies: 3706
sequence_length: 4
embedding_dim: 32
num_heads: 4
dropout_rate: 0.1


4. Create tf.data.Dataset for training and evaluation

In [33]:
# 4. Create tf.data.Dataset for training and evaluation

from sklearn.model_selection import train_test_split
import tensorflow as tf
from keras.layers import StringLookup

# Split dataset
train_data, test_data = train_test_split(
    sequence_data,
    test_size=0.2,
    random_state=42
)

# Create lookup layers
user_lookup = StringLookup(
    vocabulary=user_ids,
    mask_token=None,
    num_oov_indices=0
)

movie_lookup = StringLookup(
    vocabulary=movie_ids,
    mask_token=None,
    num_oov_indices=0
)

# ---------- TRAIN DATA ----------

train_user_ids = np.array(
    user_lookup(
        np.array(train_data["user_id"].tolist())
    )
)

train_movie_sequences = np.array(
    movie_lookup(
        np.array(train_data["movie_sequence"].tolist())
    )
)

train_rating_sequences = np.array(
    train_data["rating_sequence"].tolist(),
    dtype=np.float32
)

train_targets = np.array(
    movie_lookup(
        np.array(train_data["target_movie"].tolist())
    )
)

# ---------- TEST DATA ----------

test_user_ids = np.array(
    user_lookup(
        np.array(test_data["user_id"].tolist())
    )
)

test_movie_sequences = np.array(
    movie_lookup(
        np.array(test_data["movie_sequence"].tolist())
    )
)

test_rating_sequences = np.array(
    test_data["rating_sequence"].tolist(),
    dtype=np.float32
)

test_targets = np.array(
    movie_lookup(
        np.array(test_data["target_movie"].tolist())
    )
)

# ---------- BUILD INPUT DICTIONARIES ----------

train_inputs = {
    "user_id": train_user_ids.reshape(-1, 1).astype(np.int32),
    "movie_sequence": train_movie_sequences.astype(np.int32),
    "rating_sequence": train_rating_sequences.astype(np.float32),
}

test_inputs = {
    "user_id": test_user_ids.reshape(-1, 1).astype(np.int32),
    "movie_sequence": test_movie_sequences.astype(np.int32),
    "rating_sequence": test_rating_sequences.astype(np.float32),
}

# ---------- CREATE TF.DATA DATASETS ----------

batch_size = 256

train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_inputs, train_targets.astype(np.int32))
)

train_dataset = (
    train_dataset
    .shuffle(10000)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_inputs, test_targets.astype(np.int32))
)

test_dataset = (
    test_dataset
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

print("Training and testing datasets created successfully!")

Training and testing datasets created successfully!


5.Create model inputs.

In [34]:
# 5. Create model inputs

# User input
user_id_input = keras.Input(
    name="user_id",
    shape=(1,),
    dtype="int32"
)

# Movie history sequence input
movie_sequence_input = keras.Input(
    name="movie_sequence",
    shape=(sequence_length,),
    dtype="int32"
)

# Ratings sequence input
rating_sequence_input = keras.Input(
    name="rating_sequence",
    shape=(sequence_length,),
    dtype="float32"
)

# Store all inputs in a dictionary
inputs = {
    "user_id": user_id_input,
    "movie_sequence": movie_sequence_input,
    "rating_sequence": rating_sequence_input,
}

print("Model inputs created successfully!")
print(inputs)

Model inputs created successfully!
{'user_id': <KerasTensor shape=(None, 1), dtype=int32, sparse=False, ragged=False, name=user_id>, 'movie_sequence': <KerasTensor shape=(None, 4), dtype=int32, sparse=False, ragged=False, name=movie_sequence>, 'rating_sequence': <KerasTensor shape=(None, 4), dtype=float32, sparse=False, ragged=False, name=rating_sequence>}


6. Create a BST model.

In [35]:
# 6. Create a BST (Behavior Sequence Transformer) model

from keras import layers
import keras.ops as ops

# ---------- INPUT REFERENCES ----------

encoded_user = user_id_input
encoded_movies = movie_sequence_input

# ---------- USER EMBEDDING ----------

user_embedding = layers.Embedding(
    input_dim=num_users + 1,
    output_dim=embedding_dim
)(encoded_user)

user_embedding = layers.Flatten()(user_embedding)

# ---------- MOVIE EMBEDDINGS ----------

movie_embedding_layer = layers.Embedding(
    input_dim=num_movies + 1,
    output_dim=embedding_dim
)

movie_embeddings = movie_embedding_layer(encoded_movies)

# ---------- POSITIONAL EMBEDDINGS ----------

position_indices = ops.arange(start=0, stop=sequence_length, step=1)

position_embedding_layer = layers.Embedding(
    input_dim=sequence_length,
    output_dim=embedding_dim
)

position_embeddings = position_embedding_layer(position_indices)

# Add positional embeddings
encoded_sequence = movie_embeddings + position_embeddings

# ---------- ADD RATING INFORMATION ----------

ratings_embedding = layers.Reshape((sequence_length, 1))(
    rating_sequence_input
)

encoded_sequence = layers.Concatenate(axis=-1)(
    [encoded_sequence, ratings_embedding]
)

# Project back to embedding size
encoded_sequence = layers.Dense(embedding_dim)(
    encoded_sequence
)

# ---------- TRANSFORMER BLOCK ----------

attention_output = layers.MultiHeadAttention(
    num_heads=num_heads,
    key_dim=embedding_dim,
    dropout=dropout_rate
)(
    encoded_sequence,
    encoded_sequence
)

# Residual connection
x = layers.Add()([
    encoded_sequence,
    attention_output
])

x = layers.LayerNormalization(epsilon=1e-6)(x)

# Feed Forward Network
ffn = layers.Dense(
    embedding_dim * 2,
    activation="relu"
)(x)

ffn = layers.Dense(embedding_dim)(ffn)

# Residual connection
x = layers.Add()([x, ffn])

x = layers.LayerNormalization(epsilon=1e-6)(x)

# ---------- POOLING ----------

x = layers.GlobalAveragePooling1D()(x)

# Combine with user embedding
x = layers.Concatenate()([
    x,
    user_embedding
])

# ---------- FINAL DENSE LAYERS ----------

x = layers.Dense(128, activation="relu")(x)

x = layers.Dropout(0.2)(x)

# ---------- OUTPUT LAYER ----------

outputs = layers.Dense(
    num_movies + 1,
    activation="softmax"
)(x)

# ---------- BUILD MODEL ----------

bst_model = keras.Model(
    inputs=inputs,
    outputs=outputs
)

# ---------- MODEL SUMMARY ----------

bst_model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ movie_sequence      │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_10        │ (None, 4, 32)     │    118,624 │ movie_sequence[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rating_sequence     │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 4, 32)     │          0 │ embedding_10[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_6 (Reshape) │ (None, 4, 1)      │          0 │ rating_sequence[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, 4, 33)     │          0 │ add_9[0][0],      │
│ (Concatenate)       │                   │            │ reshape_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 4, 32)     │      1,088 │ concatenate_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 4, 32)     │     16,800 │ dense_15[0][0],   │
│ (MultiHeadAttentio… │                   │            │ dense_15[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 4, 32)     │          0 │ dense_15[0][0],   │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 4, 32)     │         64 │ add_10[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 4, 64)     │      2,112 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 4, 32)     │      2,080 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (None, 4, 32)     │          0 │ layer_normalizat… │
│                     │                   │            │ dense_17[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_id             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 4, 32)     │         64 │ add_11[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_9         │ (None, 1, 32)     │    193,312 │ user_id[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32)        │          0 │ embedding_9[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_7       │ (None, 64)        │          0 │ global_average_p

 Total params: 820,667 (3.13 MB)

 Trainable params: 820,667 (3.13 MB)

 Non-trainable params: 0 (0.00 B)

7. Run training and evaluation experiment.

In [36]:
# 7. Run training and evaluation experiment

# Compile the BST model
bst_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train the model
history = bst_model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=5
)

# Evaluate the model
test_loss, test_accuracy = bst_model.evaluate(test_dataset)

# Print results
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Epoch 1/5
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 235s 73ms/step - accuracy: 0.0119 - loss: 6.7014 - val_accuracy: 0.0203 - val_loss: 6.1970
Epoch 2/5
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 207s 59ms/step - accuracy: 0.0221 - loss: 6.0497 - val_accuracy: 0.0252 - val_loss: 5.9060
Epoch 3/5
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 202s 59ms/step - accuracy: 0.0263 - loss: 5.8230 - val_accuracy: 0.0284 - val_loss: 5.7689
Epoch 4/5
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 180s 59ms/step - accuracy: 0.0294 - loss: 5.6847 - val_accuracy: 0.0306 - val_loss: 5.6946
Epoch 5/5
3051/3051 ━━━━━━━━━━━━━━━━━━━━ 203s 59ms/step - accuracy: 0.0324 - loss: 5.5913 - val_accuracy: 0.0323 - val_loss: 5.6506
763/763 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.0323 - loss: 5.6506

Test Loss: 5.6506
Test Accuracy: 0.0323
